# 2. Data Preprocessing and Feature Engineering

This notebook implements the preprocessing and feature engineering steps approved in the pipeline plan. It imports modules from the `src` directory, processes the raw dataset, builds new features, and outputs a cleaned dataset ready for model training.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# Add src directory to path
sys.path.append(os.path.abspath('../'))

from src.data.preprocessing import preprocess_pipeline
from src.features.build_features import pipeline_feature_engineering

## 1. Execute Preprocessing Pipeline

The preprocessing pipeline handles:
- Duplicate timestamp aggregation (using mean for numeric and first for categorical).
- Outlier treatment (replacing absolute zero temperatures and extreme rainfall with NaN).
- Removing the critical missing data gap (August 2014 - June 2015).
- Reindexing the series to hourly and imputing small gaps (<= 2 hours).

In [ ]:
raw_data_path = '../data/raw/Metro_Interstate_Traffic_Volume.csv'
df_preprocessed = preprocess_pipeline(raw_data_path)

print(f"Preprocessed dataset dimensions: {df_preprocessed.shape}")
df_preprocessed.head()

### Verify Outlier Treatments

In [ ]:
# Verify absolute zero temperature values are resolved
print("Minimum temperature (K) after preprocessing:", df_preprocessed['temp'].min())
print("Maximum temperature (K) after preprocessing:", df_preprocessed['temp'].max())

# Verify extreme rain value is resolved
print("Maximum rain (mm) after preprocessing:", df_preprocessed['rain_1h'].max())

# Check remaining NaNs
print("\nRemaining missing values:")
print(df_preprocessed.isna().sum())

## 2. Execute Feature Engineering Pipeline

The feature engineering pipeline:
- Extracts temporal attributes (`hour`, `day_of_week`, `month`, `year`).
- Creates cyclical sine/cosine variables for hourly and weekly patterns.
- Derives weekend and rush-hour binary flags.
- Formulates `is_holiday` flag.
- Generates robust weather indicators (`is_raining`, `is_snowing`, `is_foggy_misty`) to address sensor quality issues.
- Computes scaling (Z-score standard scaling for temperature) and One-Hot Encoding for categorical weather.

In [ ]:
df_features, scalers = pipeline_feature_engineering(df_preprocessed)

print(f"Feature engineered dataset dimensions: {df_features.shape}")
df_features.head()

### Verify Robust Weather Flags

Let's check if the `is_snowing` indicator successfully captured snow events that the original low-quality `snow_1h` variable missed.

In [ ]:
# Cross-reference snow_1h vs. is_snowing where weather_description mentions snow
snow_check = df_features[df_features['weather_description'].str.contains('snow|sleet', case=False, na=False)]
print("Snow events description counts:")
print(snow_check['weather_description'].value_counts())
print("\nis_snowing values for those descriptions (should all be 1):")
print(snow_check['is_snowing'].value_counts())
print("\nOriginal snow_1h values for those descriptions (should have many 0s due to quality issues):")
print(snow_check['snow_1h'].value_counts().head(5))

## 3. Discard Remaining NaNs for Model Training

Since some gaps were larger than 2 hours, they were left as `NaN` (because large gaps are not imputed). Since we cannot train models on missing targets (`traffic_volume`), we will drop any row that contains missing target values.

In [ ]:
print(f"Rows before dropping rows with missing target: {len(df_features)}")
df_final = df_features.dropna(subset=['traffic_volume'])
print(f"Rows after dropping rows with missing target: {len(df_final)}")

# Check if there are any remaining NaNs in other columns
print("\nRemaining NaNs in final dataset:")
print(df_final.isna().sum().sum())

## 4. Save Processed Data and Scalers

Now we save the final cleaned and feature-engineered dataset into `data/processed/clean_traffic_data.csv`, and save the fitted scalers/encoders to `models/` for inference/testing pipeline.

In [ ]:
# Ensure directories exist
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save data
df_final.to_csv('../data/processed/clean_traffic_data.csv')
print("Processed dataset saved to '../data/processed/clean_traffic_data.csv'")

# Save encoders and scalers
with open('../models/scalers_and_encoders.pkl', 'wb') as f:
    pickle.dump(scalers, f)
print("Scalers and encoders saved to '../models/scalers_and_encoders.pkl'")